# Contextual Compression (문맥 압축)
문맥 압축은 크게 두 가지 관점으로 볼 수 있다.
1. 사전 압축 : 문서를 미리 요약하여 압축된 문서 인덱스를 만든 뒤 검색한다.
2. 사후 압축 : 원본 문서를 먼저 검색한 뒤, 검색된 문서에서 질문과 관련 있는 내용만 추출한다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

In [2]:
import pandas as pd

document_df = pd.read_csv("data/documents.csv")
queries_df = pd.read_csv("data/queries.csv")
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=2
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=2
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=2


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

def dense_search(query,top_k=5):
    """Dense retrieval로 질문과 관련있는 상위 문서 ID를 반환한다."""
    docs = vector_store.similarity_search(query,k=top_k)
    return [doc.metadata["doc_id"] for doc in docs]

## 사전 압축: 문서를 미리 요약하여 인덱싱 하기
- 검색 전에 문서 자체를 줄여놓는 방식
- 원본 문서가 길고, 반복적인 내용이 많을 때 핵심 내용만 남긴 요약 문서를 검색 대상으로 사용할 수 있다.
- 단, 특정 질문에 필요한 세부 정보가 요약 과정에서 사라질 수 있다.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_llm = ChatOpenAI(model=OPENAI_LLM_MODEL,temperature=0.2)
output_parser = StrOutputParser()

summary_prompt = PromptTemplate.from_template('''
다음 문서를 검색용 요약문으로 압축하세요.

규칙:
- 문서의 핵심 주제와 고유명사를 유지하세요.
- 검색에 중요한 키워드를 빠뜨리지 마세요.
- 불필요한 수식어와 반복 표현은 줄이세요.
- 2~3문장 이내로 작성하세요.

문서 : {text}

검색용 요약문:
''')

summary_chain = summary_prompt | summary_llm | output_parser

In [5]:
# 요약 결과 예시 확인
text = document_df.loc[0,'content']
summary = summary_chain.invoke({'text':text})

print('원본 길이 :',len(text))
print(text)
print()
print("압축본 길이 :",len(summary))
print(summary)

원본 길이 : 191
제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.

압축본 길이 : 141
제주도는 한라산 등반, 성산 일출봉, 협재·함덕해수욕장 등 관광명소와 흑돼지, 고기국수, 전복죽 등 현지 음식이 유명하며, 서귀포 대정읍 카페 거리도 인기입니다. 렌터카와 시외버스를 주로 이용하며, 사전 예약 시 우도 투어와 올레길 트레킹이 가능합니다.


## 압축 문서 생성

In [ ]:
from pathlib import Path
from tqdm import tqdm

compressed_path = Path('data/document_compression.csv')

if compressed_path.exists():
    compressed_df = pd.read_csv(compressed_path)
    print("저장된 압축 파일을 읽어봤습니다.")

else:
    compressed_texts = []

    for idx, row in tqdm(document_df.iterrows(),total = len(document_df)):
        doc_id = row['doc_id']
        content = row['content']

        summary = summary_chain.invoke({"text":content})
        compressed_texts.append({'doc_id':doc_id,'content':summary})

    compressed_df = pd.DataFrame(compressed_texts)
    compressed_df.to_csv(compressed_path,index=False)
    print("압축 문서를 저장했습니다.")

compressed_df.head()

100%|██████████| 30/30 [00:57<00:00,  1.90s/it]

압축 문서를 정장했습니다.


,doc_id,content
0,D1,"제주도는 한라산 등반, 성산 일출봉, 협재·함덕해수욕장 등 관광명소와 흑돼지, 고기..."
1,D2,"비빔밥은 조선 시대부터 전해진 한국 전통 음식으로, 전주 비빔밥은 다양한 고명과 전..."
2,D3,"걸스데이는 2010년 데뷔한 대한민국 4인조 걸그룹으로, 대표곡은 “Somethin..."
3,D4,세종대왕(1397~1450)은 백성들의 문맹 문제 해결과 국가 통치 효율화를 위해 ...
4,D5,이순신 장군(1545~1598)은 임진왜란 명량 해전에서 13척의 배로 133척의 ...
5,D6,"2024년 전 지구 평균 기온은 산업화 이전 대비 1.2℃ 상승했으며, 해수면 상승..."
6,D7,"최근 인공지능 분야에서는 GPT-4, GPT-4o-mini 등 생성형 AI와 DAL..."
7,D8,"서울 지하철 1~9호선은 서울역, 강남역, 종로3가역, 고속터미널역 등 주요 환승역..."
8,D9,"판소리는 소리꾼과 고수가 함께하는 한국 전통 음악으로, 대표 작품으로 “춘향가”, ..."
9,D10,"한국 축구 대표팀은 2002 한일 월드컵 4강, 2012 런던 올림픽 동메달, 20..."


## 압축 문서 인덱스 생성 및 저장

In [10]:
PINECONE_COMP_INDEX_NAME = 'adv-comp-rag'

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()
print(pc.list_indexes().names())

if PINECONE_COMP_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_COMP_INDEX_NAME,
        dimension=PINECONE_INDEX_DEMENSION,
        metric=PINECONE_INDEX_METRIc,
        spec=ServerlessSpec(
            region=PINECONE_INDEX_REGION,
            cloud=PINECONE_INDEX_CLOUD
        )
    )
    print(f'{PINECONE_COMP_INDEX_NAME} index 생성 완료')
else:
    print(f'{PINECONE_COMP_INDEX_NAME} index가 이미 존재합니다.')

['adv-rag', 'winemeg-review-data']
adv-comp-rag index 생성 완료


In [11]:
# 벡터 스토어 생성
comp_vector_store = PineconeVectorStore(
    index_name=PINECONE_COMP_INDEX_NAME,
    embedding=embeddings
)

In [12]:
from langchain_core.documents import Document

docs = []
ids = []

for idx,row in compressed_df.iterrows():
    doc_id = row['doc_id']
    content = row['content']
    
    doc = Document(
        page_content=content,
        metadata = {
            "doc_id":doc_id
        }
    )

    docs.append(doc)
    ids.append(doc_id)

comp_vector_store.add_documents(
    documents=docs,
    ids=ids
)

print("Pinecone 문서 저장 완료")
print(comp_vector_store._index.describe_index_stats())

Pinecone 문서 저장 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 30}},
 'total_vector_count': 30,
 'vector_type': 'dense'}


## 사전 압축 엔덱스 검색 성능 평가

In [13]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

In [14]:
dense_results = {}
compressed_results = {}

for idx, row in tqdm(queries_df.iterrows(),total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    dense_docs = vector_store.similarity_search(query_text,k=5)
    comp_docs = comp_vector_store.similarity_search(query_text,k=5)
    
    dense_results[qid] = [doc.metadata['doc_id'] for doc in dense_docs]
    compressed_results[qid] = [doc.metadata['doc_id'] for doc in comp_docs]
    

100%|██████████| 30/30 [00:34<00:00,  1.13s/it]


In [ ]:
dense_metrics = evaluate_all(dense_results,queries_df)
compressed_metrics = evaluate_all(compressed_results,queries_df)


metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'compressed' : [compressed_metrics["Precision@k"],compressed_metrics["Recall@k"],compressed_metrics["MRR"],compressed_metrics["MAP"]],
})
metrics_df

,Metric,Dense,rerank
0,Precision@k,0.233333,0.240000
1,Recall@k,0.975000,0.983333
2,MRR,1.000000,1.000000
3,MAP,0.975000,0.981667


## 사후 압축: 검색된 문서를 질문 기준으로 압축하기
- 검색 결과 문서 ID를 바꾸는 것이 아니라 LLM에게 전달되는 context의 길이와 노이즈를 줄이는 목적

In [16]:
compression_prompt = PromptTemplate.from_template("""
다음 문서에서 사용자 질문에 답하는데 필요한 내용만 추출하세요.
                                                  
규칙:
- 문서에 질문과 관련있는 내용이 있으면 관련 문장만 간결하게 추출하세요.
- 질문과 관련 없는 배경 설명은 제거하세요.
- 문서에 질문과 관련 있는 내용이 거의 없으면 "관련 내용 없음"이라고 작성하세요.
- 새 정보를 지어내지 마세요.
                                                  
사용자 질문:
{query}
                                                  
문서:
{document}
                                                  
압축된 문맥:
""")

compression_llm = ChatOpenAI(model=OPENAI_LLM_MODEL,temperature=0)
compression_chain = compression_prompt | compression_llm | output_parser

In [ ]:
def compress_retrieved_docs(query,docs):
    """검색된 문서를 질문 기준으로 압축한다."""
    compressed_texts = []

    for doc in docs:
        compressed_text = compression_chain.invoke({
            'query':query,
            'document':doc.page_content
        })

        compressed_texts.append({
            'doc_id':doc.metadata['doc_id'],
            'original_text':doc.page_content,
            'compressed_text':compressed_text,
            'original_length':len(doc.page_content),
            'compressed_length':len(compressed_text)    
        })

    return compressed_texts

In [ ]:
sample_query = '세종대왕이 훈민정음을 만든 배경은?'

retrieved_docs = vector_store.similarity_search(sample_query, k=5)
compressed_contexts = compress_retrieved_docs(sample_query,retrieved_docs)

for item in compressed_contexts:
    print(item['doc_id'])
    print('원본 길이 :', item['original_length'])
    print('압축 길이 :', item['compressed_length'])
    print()
    print('압축된 문맥')
    print(item['compressed_text'])
    print("-"*100)

D4
원본 길이 : 170
압축 길이 : 52

압축된 문맥
세종대왕이 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다.
----------------------------------------------------------------------------------------------------
D30
원본 길이 : 250
압축 길이 : 8

압축된 문맥
관련 내용 없음
----------------------------------------------------------------------------------------------------
D5
원본 길이 : 164
압축 길이 : 8

압축된 문맥
관련 내용 없음
----------------------------------------------------------------------------------------------------
D18
원본 길이 : 236
압축 길이 : 8

압축된 문맥
관련 내용 없음
----------------------------------------------------------------------------------------------------
D3
원본 길이 : 193
압축 길이 : 8

압축된 문맥
관련 내용 없음
----------------------------------------------------------------------------------------------------
